In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_interest_future_spread_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema=(
        #    "ExposureInterestFutureSpreadId string,"
        #    "SourceSystem string,"
        #    "PortfolioKey string,"
        #    "PortfolioName string,"
        #    "IngestionTimestamp timestamp,"
        #    "BatchId string,"
        #    "ExposureInterestFutureSpreadKey string,"
        #    "ExposureKey string,"
        #    "ExposureUnit decimal(5,0),"
        #    "ExpsosureType decimal(2,0),"
        #    "ExposureSN decimal(6,0),"
        #    "FutureMarginApplyDate timestamp,"
        #    "CurrentUnit decimal(5,0),"
        #    "UpdatedUnit decimal(5,0),"
        #    "DebitSpreadRate decimal(8,4),"
        #    "CreditSpread decimal(8,4),"
        #    "CreatedBy string,"
        #    "CreationDate timestamp,"
        #    "UpdatedBy string,"
        #    "LastUpdatedDate timestamp,"
        #    "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExposureInterestFutureSpread"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")
        # ================== profits_loan_details ==================

        loan_details_extra_filters = None

        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter

        loan_details_selected_fields = [
            "ACCOUNT_NUMBER",
            "prft_acc_unit",
            "prft_acc_type",
            "prft_acc_sn",
        ]

        loan_details = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        )

        # ==================== profits_loan_int_spread ==============

        loan_int_spread_extra_filters = None

        if loan_int_spread_extra_filters is not None:
            loan_int_spread_extra_filters.append(date_filter)
            loan_int_spread_final_filters = " AND ".join(loan_int_spread_extra_filters)
        else:
            loan_int_spread_final_filters = date_filter

        loan_int_spread_selected_fields = [
            "FK_LOAN_ACCOUNTACC",
            "FK0LOAN_ACCOUNTACC",
            "FK_LOAN_ACCOUNTFK",
            "EFFECTIVE_DATE",
            "FK_LOAN_ACCOUNTFK",
            "FK0LOAN_ACCOUNTACC",
            "FK_LOAN_ACCOUNTACC",
            "Unit",
            "UPDATE_UNIT",
            "DB_INTER_RATE_SPR",
            "CRINT_SPREAD",
            "USR",
            "CREATE_TMSTMP",
            "UPDATE_USER",
            "UPDATE_TMSTMP",
            "BusinessDate",
        ]

        loan_int_spread = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_int_spread",
            loan_int_spread_final_filters,
            loan_int_spread_selected_fields,
        )

        # ==================== portfolio =====================

        portfolio_spread_extra_filters = None

        if portfolio_spread_extra_filters is not None:
            portfolio_spread_extra_filters.append(date_filter)
            portfolio_spread_final_filters = " AND ".join(
                portfolio_spread_extra_filters
            )
        else:
            portfolio_spread_final_filters = date_filter

        portfolio_spread_selected_fields = ["PortfolioName", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_spread_final_filters,
            portfolio_spread_selected_fields,
        )

        # ====================== joins ==========================
        loan_details_loan_int_spread_join_condition = (
            (loan_details.prft_acc_sn == loan_int_spread.FK_LOAN_ACCOUNTACC)
            & (loan_details.prft_acc_type == loan_int_spread.FK0LOAN_ACCOUNTACC)
            & (loan_details.prft_acc_unit == loan_int_spread.FK_LOAN_ACCOUNTFK)
        )

        portfolio_loan_details_join_condition = (
            portfolio.unit_code_profits == loan_details.prft_acc_unit
        )

        exposure_interest_df = loan_int_spread.join(
            loan_details,
            loan_details_loan_int_spread_join_condition,
            "inner",
        ).join(
            portfolio,
            portfolio_loan_details_join_condition,
            "left",
        )

        ExposureInterestFutureSpreadId_key_fields = [
            exposure_interest_df.FK_LOAN_ACCOUNTACC,
            exposure_interest_df.FK0LOAN_ACCOUNTACC,
            exposure_interest_df.FK_LOAN_ACCOUNTFK,
            exposure_interest_df.EFFECTIVE_DATE,
            lit("profits"),
        ]

        ExposureInterestFutureSpreadKey_key_fields = [
            exposure_interest_df.FK_LOAN_ACCOUNTACC,
            exposure_interest_df.FK0LOAN_ACCOUNTACC,
            exposure_interest_df.FK_LOAN_ACCOUNTFK,
            exposure_interest_df.EFFECTIVE_DATE,
        ]

        exposure_interest_df = exposure_interest_df.withColumn(
            "ExposureInterestFutureSpreadId",
            sha2(
                concat_ws(";", *ExposureInterestFutureSpreadId_key_fields),
                256,
            ),
        ).withColumn(
            "ExposureInterestFutureSpreadKey",
            concat_ws(";", *ExposureInterestFutureSpreadKey_key_fields),
        )

        result_df = exposure_interest_df.select(
            exposure_interest_df.ExposureInterestFutureSpreadId.cast("string").alias(
                "ExposureInterestFutureSpreadId"
            ),
            lit("profits").alias("SourceSystem"),
            exposure_interest_df.unit_code_profits.cast("string").alias("PortfolioKey"),
            exposure_interest_df.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            exposure_interest_df.ExposureInterestFutureSpreadKey.cast("string").alias(
                "ExposureInterestFutureSpreadKey"
            ),
            exposure_interest_df.ACCOUNT_NUMBER.cast("string").alias("ExposureKey"),
            exposure_interest_df.FK_LOAN_ACCOUNTFK.alias("ExposureUnit"),
            exposure_interest_df.FK0LOAN_ACCOUNTACC.alias("ExpsosureType"),
            exposure_interest_df.FK_LOAN_ACCOUNTACC.alias("ExposureSN"),
            exposure_interest_df.EFFECTIVE_DATE.alias("FutureMarginApplyDate"),
            exposure_interest_df.Unit.alias("CurrentUnit"),
            exposure_interest_df.UPDATE_UNIT.alias("UpdatedUnit"),
            exposure_interest_df.DB_INTER_RATE_SPR.alias("DebitSpreadRate"),
            exposure_interest_df.CRINT_SPREAD.alias("CreditSpread"),
            exposure_interest_df.USR.alias("CreatedBy"),
            exposure_interest_df.CREATE_TMSTMP.alias("CreationDate"),
            exposure_interest_df.UPDATE_USER.alias("UpdatedBy"),
            exposure_interest_df.UPDATE_TMSTMP.alias("LastUpdatedDate"),
            exposure_interest_df.BusinessDate.alias("BusinessDate"),
        )
        # ========================== Append ==========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_exposure_interest_future_spread_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)